# Read and plot the original source

Same complete `Fig_4g_OD600` readout and plotting options as `read_atst.ipynb`. Source ranges and condition annotations remain explicit; no ATST file is read. Matplotlib plots the source values directly.


In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd
from openpyxl import load_workbook

readout_id = 'Fig_4g_OD600'
rows, curves = [], []

sheet = load_workbook("read_shared_data/sources/41467_2022_31934_MOESM7_ESM.xlsx", read_only=True, data_only=True)["Fig. 4g"]
condition = ""
for r in range(1, sheet.max_row + 1):
    value = sheet.cell(r, 7).value
    if isinstance(value, str) and "phage cocktail" in value:
        condition = value
    if value != "Time (h)":
        continue
    data_rows = []
    k = r + 1
    while k <= sheet.max_row and isinstance(sheet.cell(k, 7).value, (int, float)):
        data_rows.append(k)
        k += 1
    if not data_rows:
        continue
    phage = "phage_cocktail" if "with phage cocktail" in condition and "without phage cocktail" not in condition else ""
    isolate = "MG1655DE3_EPR" if "EPR" in condition else "MG1655DE3_WT"
    time = [sheet.cell(i, 7).value for i in data_rows]
    for replicate in range(1, 3):
        rows.append(dict(type="phage_exposed" if phage else "uninfected_control", isolate_id=isolate, phage_id=phage, replicate=str(replicate)))
        curves.append(pd.Series([sheet.cell(i, 7 + replicate).value for i in data_rows], index=time))

# Local curve IDs preserve source order; different time grids align without interpolation.
layout = pd.DataFrame(rows)
layout["curve_id"] = [f"curve_{i}" for i in range(1, len(rows) + 1)]
readings = pd.concat(curves, axis=1).sort_index()
readings.columns = layout["curve_id"]
readings = readings.rename_axis("Time").reset_index()


In [ ]:
panels = list(layout.groupby('isolate_id', sort=False))
ncols = min(2, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = {}

for ax, (panel, panel_layout) in zip(axes.flat, panels):
    for condition, group in panel_layout.groupby('phage_id', sort=False):
        if condition not in condition_colors:
            condition_colors[condition] = palette[len(condition_colors) % len(palette)]
        color = condition_colors[condition]
        label = str(condition)
        values = readings[group["curve_id"]]
        for index, curve_id in enumerate(group["curve_id"]):
            ax.plot(
                readings["Time"], readings[curve_id], color=color, alpha=0.6,
                label=label if index == 0 else None,
            )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("od600")
    panel_label = "<blank>" if panel == "" else str(panel)
    ax.set_title(f"{readout_id} | isolate_id={panel_label}")
    ax.legend(frameon=False)

for ax in list(axes.flat)[len(panels):]:
    fig.delaxes(ax)
fig.tight_layout()
plt.show()


In [ ]:
panels = list(layout.groupby('isolate_id', sort=False))
ncols = min(2, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = {}

for ax, (panel, panel_layout) in zip(axes.flat, panels):
    for condition, group in panel_layout.groupby('phage_id', sort=False):
        if condition not in condition_colors:
            condition_colors[condition] = palette[len(condition_colors) % len(palette)]
        color = condition_colors[condition]
        label = str(condition)
        values = readings[group["curve_id"]]
        median = values.median(axis=1)
        minimum = values.min(axis=1)
        maximum = values.max(axis=1)
        ax.errorbar(
            readings["Time"], median,
            yerr=[median - minimum, maximum - median],
            fmt="o-", capsize=3, color=color, label=label,
        )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("od600")
    panel_label = "<blank>" if panel == "" else str(panel)
    ax.set_title(f"{readout_id} | isolate_id={panel_label}")
    ax.legend(frameon=False)

for ax in list(axes.flat)[len(panels):]:
    fig.delaxes(ax)
fig.tight_layout()
plt.show()
